In [106]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm
from arch import arch_model
from arch.unitroot import PhillipsPerron
from mgarch import mgarch

In [107]:
from garch_functions import (
    fit_all_garch,
    diagnostics_summary_table,
    parameter_significance_table,
    plot_all_diagnostics,
    GARCHConfig,
    DEFAULT_CONFIG
)

df = pd.read_csv("data/merged_data_v2.csv", index_col='date')
df.dropna(inplace=True)
df.index= pd.to_datetime(df.index, format = '%Y-%m-%d')
weekly_df = df[["btc_adj_close", "eth_adj_close", "sp500_adj_close", "vti_adj_close", "agg_adj_close", "tlt_adj_close"]].resample("W-FRI").last().dropna()
weekly_df.columns = ["BTC", "ETH", "SP500", "VTI", "AGG", "TLT"]
price_assets = ["BTC", "ETH", "SP500", "VTI", "AGG", "TLT"] # treasury yield is in rates, will be treated differently
weekly_logret = 100*np.log(weekly_df[price_assets] / weekly_df[price_assets].shift(1)).dropna()

garch_results = fit_all_garch(weekly_logret, model_type="auto")
diag_table = diagnostics_summary_table(garch_results)
cond_vol_df = pd.DataFrame({
    asset: garch_results[asset]["cond_vol"]
    for asset in garch_results
}).dropna()


GARCH fit: BTC

  --------------------------------------------------
  AR(1) mean detected (LB p=0.0843) -- using AR(1) mean equation

  Model competition [primary=skewt, mean=AR]:
  Model            Dist            AIC        BIC     Status
  --------------------------------------------------------
  GARCH            skewt       3019.33    3047.68         ok
  GJR              skewt       3013.32    3045.72         ok
  EGARCH           skewt            --         --   no convergence -> retrying with GED
  EGARCH           ged         3002.10    3030.45 GED fallback

  -> BIC selects: EGARCH-GED  (BIC=3030.45)
  EGARCH |beta|=0.9996  ->  stationary  |  near-IGARCH -- shocks near-permanent (half-life ~1574 weeks)

  Post-fit [EGARCH-GED]:
  Ljung-Box resid   p@10=0.7007  p@20=0.6338
  Ljung-Box resid^2 p@10=0.4766  p@20=0.4628
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Lib

In [109]:
#Check Stationarity (ADF)
from statsmodels.tsa.stattools import adfuller
import pandas as pd

results = []

for col in cond_vol_df.columns:
    adf_result = adfuller(cond_vol_df[col])

    test_stat = adf_result[0]
    p_value = adf_result[1]

    if p_value < 0.01:
        sig = "***"
    elif p_value < 0.05:
        sig = "**"
    elif p_value < 0.10:
        sig = "*"
    else:
        sig = ""

    results.append([col, test_stat, p_value, sig])

df_adf = pd.DataFrame(results,
                      columns=["Asset", "ADF Statistic", "p-value", "Sig"])

df_adf["ADF Statistic"] = df_adf["ADF Statistic"].round(4)
df_adf["p-value"] = df_adf["p-value"].round(6)

df_adf

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


,Asset,ADF Statistic,p-value,Sig
0,BTC,-1.4395,0.563192,
1,ETH,-2.7157,0.071364,*
2,SP500,-6.2239,0.000000,***
3,VTI,-6.2222,0.000000,***
4,AGG,-5.4366,0.000003,***
5,TLT,-4.0378,0.001225,***


In [110]:
# BTC and ETH → log returns
for col in ['BTC', 'ETH']:
    cond_vol_df[col] = np.log(cond_vol_df[col]).diff()

# Scale BTC and ETH log returns to % change
cond_vol_df['BTC'] = cond_vol_df['BTC'] * 100
cond_vol_df['ETH'] = cond_vol_df['ETH'] * 100

# Only drop rows where BTC/ETH diff created NaN
cond_vol_df.dropna(subset=['BTC', 'ETH'], inplace=True)

cond_vol_df

,BTC,ETH,SP500,VTI,AGG,TLT
date,,,,,,
2017-12-01,0.906972,1.967053,1.974383,1.938314,0.378420,1.283179
2017-12-08,0.499348,-1.423262,1.802225,1.782683,0.373350,1.268553
2017-12-15,0.294542,-1.409209,1.683607,1.660261,0.358625,1.256977
2017-12-22,0.941224,3.012349,1.572070,1.553093,0.363641,1.323935
2017-12-29,-4.975950,-1.428020,1.490367,1.463225,0.456707,1.530994
...,...,...,...,...,...,...
2025-12-05,0.590709,-0.084108,2.020121,2.079536,0.463404,1.437080
2025-12-12,-0.727180,-0.812271,1.861121,1.907460,0.501521,1.520447
2025-12-19,0.998121,-0.776584,1.926548,1.920138,0.476537,1.511583


In [111]:
#Check Stationarity (ADF)
from statsmodels.tsa.stattools import adfuller
import pandas as pd

results = []

for col in cond_vol_df.columns:
    adf_result = adfuller(cond_vol_df[col])

    test_stat = adf_result[0]
    p_value = adf_result[1]

    if p_value < 0.01:
        sig = "***"
    elif p_value < 0.05:
        sig = "**"
    elif p_value < 0.10:
        sig = "*"
    else:
        sig = ""

    results.append([col, test_stat, p_value, sig])

df_adf = pd.DataFrame(results,
                      columns=["Asset", "ADF Statistic", "p-value", "Sig"])

df_adf["ADF Statistic"] = df_adf["ADF Statistic"].round(4)
df_adf["p-value"] = df_adf["p-value"].round(6)

df_adf

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


,Asset,ADF Statistic,p-value,Sig
0,BTC,-21.1111,0.000000,***
1,ETH,-21.3208,0.000000,***
2,SP500,-6.2219,0.000000,***
3,VTI,-6.2209,0.000000,***
4,AGG,-5.4412,0.000003,***
5,TLT,-4.0535,0.001155,***


In [112]:
#VECTOR AUTOREGRESSION (VAR)
cond_vol_df.index = pd.to_datetime(cond_vol_df.index)

from statsmodels.tsa.api import VAR

model = VAR(cond_vol_df)
lag_selection = model.select_order(maxlags=8)
print(lag_selection.summary())

results = model.fit(lag_selection.aic)
print(results.summary())

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [113]:
#GRANGER CAUSALITY TEST
from statsmodels.tsa.stattools import grangercausalitytests
import itertools
import pandas as pd

max_lag = 1 #BIC
alpha = 0.05

results_list = []

columns = cond_vol_df.columns

for y_col, x_col in itertools.permutations(columns, 2):
    test_result = grangercausalitytests(cond_vol_df[[y_col, x_col]], maxlag=max_lag, verbose=False)

    f_stat = test_result[max_lag][0]['ssr_ftest'][0]
    p_value = test_result[max_lag][0]['ssr_ftest'][1]

    # significance stars
    if p_value < 0.01:
        sig = "***"
    elif p_value < 0.05:
        sig = "**"
    elif p_value < 0.10:
        sig = "*"
    else:
        sig = ""

    results_list.append([x_col, y_col, f_stat, p_value, sig])

df_results = pd.DataFrame(results_list, columns=["Cause", "Target", "F-stat", "p-value", "Sig"])

print(df_results)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [114]:
from statsmodels.tsa.stattools import grangercausalitytests
import itertools
import pandas as pd

max_lag = 3 #FPE & AIC
alpha = 0.05

results_list = []

columns = cond_vol_df.columns

for y_col, x_col in itertools.permutations(columns, 2):
    test_result = grangercausalitytests(cond_vol_df[[y_col, x_col]], maxlag=max_lag, verbose=False)

    f_stat = test_result[max_lag][0]['ssr_ftest'][0]
    p_value = test_result[max_lag][0]['ssr_ftest'][1]

    # significance stars
    if p_value < 0.01:
        sig = "***"
    elif p_value < 0.05:
        sig = "**"
    elif p_value < 0.10:
        sig = "*"
    else:
        sig = ""

    results_list.append([x_col, y_col, f_stat, p_value, sig])

df_results = pd.DataFrame(results_list, columns=["Cause", "Target", "F-stat", "p-value", "Sig"])

print(df_results)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [41]:
from statsmodels.tsa.stattools import grangercausalitytests
import itertools

max_lag = 4
alpha = 0.05

# Loop through all combinations of columns (Y, X) where Y != X
columns = cond_vol_df.columns
for y_col, x_col in itertools.permutations(columns, 2):
    print(f"\nTesting if '{x_col}' Granger-causes '{y_col}':")
    results = grangercausalitytests(cond_vol_df[[y_col, x_col]], maxlag=max_lag, verbose=False)

    for lag in range(1, max_lag + 1):
        f_test_p = results[lag][0]['ssr_ftest'][1]  # p-value of SSR F-test
        significance = "Significant" if f_test_p < alpha else "Not significant"
        symbol = "<" if f_test_p < alpha else ">"
        print(f"  Lag {lag}: p = {f_test_p:.4f} {symbol} {alpha} → {significance}")


Testing if 'ETH' Granger-causes 'BTC':
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector 